# Global Daily Event Analysis: Marine Heatwave ID & Tracking using `MarEx`

### `MarEx` Processing Pipeline for Unstructured Datasets:

1. **Morphological Pre-Processing**
    - Performs binary morphological closing using highly-threaded binary dilation matrix operations to fill small spatial holes up to `R_fill` elements in radius 
    - Executes binary opening to remove isolated small features of order `R_fill`
    - Fills gaps in time to maintain event continuity for interruptions up to `T_fill` time steps
    - Filters out smallest objects below the `area_filter_quartile` percentile threshold

2. **Blob Identification**
    - Labels spatially connected components using a highly efficient Unstructured Union-Find (Disjoint Set Union) Clustering Algorithm
    - Computes blob properties (area, centroid, boundaries)

3. **Temporal Tracking**
    - Identifies blob overlaps between consecutive time frames
    - Connects objects across time, applying the following criteria for splitting, merging, & persistence:
        - Connected objects must overlap by at least fraction `overlap_threshold` of the smaller area
        - Merged objects retain their original ID, but partition the child area based on the parent of the _nearest-neighbour_ cell (or centroid distance)

4. **Graph Reduction & Finalisation**
    - Constructs the complete temporal graph of object evolution through time
    - Resolves object connectivity graph using `scipy.sparse.csgraph.connected_components`
    - Creates globally unique IDs for each tracked extreme event
    - Maps objects into efficient ID-time space for convenient analysis
    - Computes comprehensive statistics about the lifecycle of each event

The pipeline leverages **dask** for distributed parallel computation, enabling efficient processing of large datasets. \
A 40-year global daily analysis at 5km resolution on the _unstructured grid_ (15 million cells) using 240 cores takes ~40 minutes.

#### N.B.: The following `dask` config may be necessary on particular systems:
```python
dask.config.set({
    'distributed.comm.timeouts.connect': '120s',  # Increase from default
    'distributed.comm.timeouts.tcp': '240s',      # Double the connection timeout
    'distributed.comm.retry.count': 10,           # More retries before giving up
})
```

In [1]:
from getpass import getuser
from pathlib import Path

import dask
import xarray as xr

import marEx
import marEx.helper as hpc

2026-08-22 11:47:41 - marEx - INFO - [PID:2330325] - MarEx logging configured - Level: INFO, Mode: normal


In [2]:
# Lustre Scratch Directory
scratch_dir = Path("/scratch") / getuser()[0] / getuser()

In [3]:
# # Start Distributed Dask Cluster
# client_cluster = hpc.start_distributed_cluster(n_workers=2048, workers_per_node=128, runtime=59, node_memory=256,
#                                  scratch_dir = scratch_dir / 'clients')  # Specify temporary scratch directory for dask to use

# N.B. `memory_limit` is set EXPLICITLY. dask is cgroup-blind here: LocalCluster sizes its
# workers from the node's total RAM rather than from this job's allocation, so without it the
# workers below would each claim ~1/n of the whole node and over-commit. 16 workers x 12 GB
# = 192 GB is the configuration measured to carry the full 1096-day record through the merge
# loop end to end (54 min wall, peak 116.8 GB, zero worker restarts).
client = hpc.start_local_cluster(
    n_workers=16, threads_per_worker=1, memory_limit="12GB", scratch_dir=scratch_dir / "clients"
)  # Specify temporary scratch directory for dask to use

2026-08-22 11:47:43 - marEx.helper.cluster - INFO - [PID:2330325] - Starting local Dask cluster with 16 workers, 1 threads each


2026-08-22 11:47:43 - marEx.helper.dask_config - INFO - [PID:2330325] - Configuring Dask for HPC environment


2026-08-22 11:47:43 - marEx.helper.dask_config - INFO - [PID:2330325] - Dask temporary directory: /scratch/b/b382615/clients/tmpguu7o_lk


2026-08-22 11:47:43 - marEx.helper.dask_config - INFO - [PID:2330325] - Dask configuration completed


2026-08-22 11:47:43 - marEx.helper.cluster - INFO - [PID:2330325] - System resources: 128 physical cores, 256 logical cores, 1007.6GB total memory


2026-08-22 11:47:43 - marEx.helper.cluster - INFO - [PID:2330325] - Memory per worker: 62.98 GB


2026-08-22 11:47:43 - marEx.helper.cluster - INFO - [PID:2330325] - Starting Local cluster startup


2026-08-22 11:47:46 - marEx.helper.cluster - INFO - [PID:2330325] - Completed Local cluster startup in 2.87s


Hostname: l40359
Forward Port: l40359:8787
Dashboard Link: localhost:8787/status
2026-08-22 11:47:46 - marEx.helper.cluster - INFO - [PID:2330325] - Local cluster started successfully - Dashboard: http://127.0.0.1:8787/status


2026-08-22 11:47:46 - marEx.helper.cluster - INFO - [PID:2330325] - After cluster startup - Memory Usage - RSS: 572.4MB, Virtual: 25677.4MB, Percent: 0.1%, Available: 1015355.0MB


In [4]:
# Choose optimal chunk size & load data
#   N.B.: This is crucial for dask (not only for performance, but also to make the problem tractable)
#         The operations are eventually global-in-space, and so requires the spatial dimension to be contiguous/unchunked
#         We can adjust the chunk size in time depending on available system memory; however,
#         note that the performance of the parallel iterative merging algorithm increases with larger chunks in time.

chunk_size = {"time": 4, "ncells": -1}

In [5]:
# Load Pre-processed Data (cf. `01_preprocess_extremes.ipynb`)
#
# N.B. chunk_size above keeps `ncells` WHOLE on purpose. Unlike notebook 01, the
# tracker is global-in-space (connected-component labelling and the dilation matrix
# span the mesh), so the spatial dimension must not be chunked here. Time is the
# lever instead -- and the iterative merging algorithm prefers larger time chunks.

file_name = scratch_dir / "mhws" / "extremes_binary_unstruct_shifting_hobday.zarr"
ds = xr.open_zarr(str(file_name), chunks=chunk_size)

# To rehearse the chain cheaply, subset deliberately and say so, e.g.
#   ds = xr.open_zarr(str(file_name), chunks=chunk_size).isel(time=slice(0, 256))
# (a hardcoded slice used to live on the line above, which silently tracked only the
#  first 256 days while the notebook claimed to analyse the full record).
ds

<xarray.Dataset> Size: 104GB
Dimensions:         (ncells: 14886338, time: 1096, dayofyear: 366, nv: 3)
Coordinates:
  * dayofyear       (dayofyear) uint16 732B 1 2 3 4 5 6 ... 362 363 364 365 366
    lat             (ncells) float64 119MB dask.array<chunksize=(14886338,), meta=np.ndarray>
    lon             (ncells) float64 119MB dask.array<chunksize=(14886338,), meta=np.ndarray>
  * nv              (nv) int64 24B 0 1 2
  * time            (time) datetime64[ns] 9kB 2012-01-01T23:59:59 ... 2014-12...
Dimensions without coordinates: ncells
Data variables:
    cell_areas      (ncells) float32 60MB dask.array<chunksize=(14886338,), meta=np.ndarray>
    dat_anomaly     (time, ncells) float32 65GB dask.array<chunksize=(4, 14886338), meta=np.ndarray>
    extreme_events  (time, ncells) bool 16GB dask.array<chunksize=(4, 14886338), meta=np.ndarray>
    mask            (ncells) bool 15MB dask.array<chunksize=(14886338,), meta=np.ndarray>
    neighbours      (nv, ncells) int32 179MB dask.array<chunksize=(3, 14886338), meta=np.ndarray>
    thresholds      (ncells, dayofyear) float32 22GB dask.array<chunksize=(14886338, 2), meta=np.ndarray>
Attributes:
    max_anomaly:           5.0
    method_anomaly:        shifting_baseline
    method_extreme:        hobday_extreme
    method_percentile:     approximate
    precision:             0.01
    preprocessing_steps:   ['Rolling climatology using 5 years', 'Smoothed wi...
    smooth_days_baseline:  21
    threshold_percentile:  95
    window_days_hobday:    11
    window_year_baseline:  5

## Identification, Tracking, & Merging

Everything runs on the single cluster started above.

An earlier version of this notebook split the work across two differently-sized clusters --
a wide one for the morphological pre-processing, a memory-heavy one for ID/tracking -- with a
checkpoint written to disk in between. That is no longer needed: under
`compute_mode="streaming"` the whole-field intermediates are staged to `temp_dir` instead of
being pinned in cluster RAM, so one cluster of 16 workers x 12 GB carries both stages of the
full 1096-day record.


In [6]:
# Run ID, Tracking, & Merging

tracker = marEx.tracker(
    ds.extreme_events,
    ds.mask,
    area_filter_absolute=13500,  # Keep only objects of at least 13500 cells (~328,000 km2 at this mesh's 24.34 km2 mean cell area). An absolute floor rather than a quartile: at 14.9M cells the smallest-80% cut still leaves tens of thousands of fragments, which fragment merge events rather than resolving them.
    R_fill=32,  # Fill small holes with radius < 32 elements, i.e. ~158 km (32 x the 4.93 km mean cell spacing),
    T_fill=2,  # Allow gaps of 2 days and still continue the event tracking with the same ID
    allow_merging=True,  # Allow extreme events to split/merge. Keeps track of merge events & unique IDs.
    overlap_threshold=0.5,  # Overlap threshold for merging events. If overlap < threshold, events keep independent IDs.
    nn_partitioning=True,  # Use new NN method to partition merged children areas. If False, reverts to old method of Di Sun et al. 2023.
    temp_dir=str(scratch_dir / "mhws" / "TEMP/"),  # Temporary Scratch Directory needed for Dask
    verbose=True,  # Enable detailed logging
    # -- Unstructured Grid Options --
    unstructured_grid=True,  # Use Unstructured Grid
    dimensions={"x": "ncells"},  # Need to tell MarEx the new Unstructured dimension
    coordinates={"x": "lon", "y": "lat"},  # Coordinates for Unstructured Grid
    neighbours=ds.neighbours,  # Connectivity array for the Unstructured Grid Cells
    cell_areas=ds.cell_areas,  # Cell areas for each Unstructured Grid Cell
    compute_mode="streaming",  # Stage every whole-field intermediate to zarr under `temp_dir` rather than pinning it in cluster RAM. Required at this length: one int32 whole field is 1096 x 14.9M x 4 B = 65.3 GB, and `persist` holds several at once.
)

2026-08-22 11:47:47 - marEx - INFO - [PID:2330325] - configure_logging:177 - MarEx logging configured - Level: DEBUG, Mode: verbose


2026-08-22 11:47:47 - marEx.track.tracker - INFO - [PID:2330325] - __init__:481 - Initialising MarEx tracker


2026-08-22 11:47:47 - marEx.track.tracker - INFO - [PID:2330325] - __init__:482 - Grid type: unstructured


2026-08-22 11:47:47 - marEx.track.tracker - INFO - [PID:2330325] - __init__:483 - Parameters: R_fill=32, T_fill=2, area_filter_quartile=None, area_filter_absolute=13500


2026-08-22 11:47:47 - marEx.track.tracker - DEBUG - [PID:2330325] - __init__:487 - Tracking options: allow_merging=True, nn_partitioning=True, overlap_threshold=0.5


...

2026-08-22 11:47:49 - marEx.detect.compute_mode - INFO - [PID:2330325] - create_staging_dir:84 - Streaming staging directory: /scratch/b/b382615/mhws/TEMP/marex_stage_2330325_b767d5c3


2026-08-22 11:47:49 - marEx.track.tracker - DEBUG - [PID:2330325] - __init__:663 - Dimensions: time=time, x=ncells, y=lat


2026-08-22 11:47:49 - marEx.track.tracker - DEBUG - [PID:2330325] - __init__:664 - Coordinates: time=time, x=lon, y=lat


2026-08-22 11:47:53 - marEx.track.grid - INFO - [PID:2330325] - build_sparse_dilation_matrix:274 - Finished constructing the sparse dilation matrix


2026-08-22 11:47:54 - marEx.track.tracker - DEBUG - [PID:2330325] - _configure_warnings:822 - Configuring warnings and logging for debug level: 0


In [7]:
# Pre-processes the binary data, then identifies, tracks and merges the events.
extreme_events_ds, merges_ds = tracker.run(return_merges=True)
extreme_events_ds

2026-08-22 11:47:54 - marEx.track.tracker - INFO - [PID:2330325] - run:893 - Starting complete tracking pipeline


2026-08-22 11:47:54 - marEx.track.tracker - DEBUG - [PID:2330325] - log_memory_usage:287 - Pipeline start - Memory Usage - RSS: 1764.9MB, Virtual: 65998.1MB, Percent: 0.2%, Available: 1013358.6MB


2026-08-22 11:47:54 - marEx.track.tracker - INFO - [PID:2330325] - run:902 - Step 1/3: Data preprocessing


2026-08-22 11:47:54 - marEx.track.tracker - DEBUG - [PID:2330325] - log_timing:318 - Initializing Data preprocessing


2026-08-22 11:47:54 - marEx.track.tracker - DEBUG - [PID:2330325] - log_memory_usage:287 - Before Data preprocessing - Memory Usage - RSS: 1764.9MB, Virtual: 65998.1MB, Percent: 0.2%, Available: 1013390.1MB


...

2026-08-22 13:01:19 - marEx.track.tracker - INFO - [PID:2330325] - run:928 - Tracking pipeline completed successfully - 4359 events identified


2026-08-22 13:01:19 - marEx.track.tracker - DEBUG - [PID:2330325] - run:929 - Final dataset dimensions: FrozenMappingWarningOnValuesAccess({'time': 1096, 'ncells': 14886338, 'ID': 4359, 'component': 2, 'sibling_ID': 12})


2026-08-22 13:01:19 - marEx.track.tracker - DEBUG - [PID:2330325] - log_memory_usage:287 - Pipeline completion - Memory Usage - RSS: 2845.5MB, Virtual: 67007.1MB, Percent: 0.3%, Available: 982888.8MB


2026-08-22 13:01:19 - marEx.track.tracker - DEBUG - [PID:2330325] - run:941 - Returning both events and merge datasets


<xarray.Dataset> Size: 66GB
Dimensions:       (time: 1096, ncells: 14886338, ID: 4359, component: 2,
                   sibling_ID: 12)
Coordinates:
  * time          (time) datetime64[ns] 9kB 2012-01-01T23:59:59 ... 2014-12-3...
  * ID            (ID) int32 17kB 1 2 3 4 5 6 ... 4354 4355 4356 4357 4358 4359
  * component     (component) int64 16B 0 1
    lat           (ncells) float64 119MB dask.array<chunksize=(14886338,), meta=np.ndarray>
    lon           (ncells) float64 119MB dask.array<chunksize=(14886338,), meta=np.ndarray>
Dimensions without coordinates: ncells, sibling_ID
Data variables:
    ID_field      (time, ncells) int32 65GB dask.array<chunksize=(1, 14886338), meta=np.ndarray>
    global_ID     (time, ID) int32 19MB dask.array<chunksize=(1, 4359), meta=np.ndarray>
    area          (time, ID) float32 19MB dask.array<chunksize=(1, 4359), meta=np.ndarray>
    centroid      (component, time, ID) float32 38MB dask.array<chunksize=(1, 1, 4359), meta=np.ndarray>
    presence      (time, ID) bool 5MB dask.array<chunksize=(1, 4359), meta=np.ndarray>
    time_start    (ID) datetime64[ns] 35kB dask.array<chunksize=(4359,), meta=np.ndarray>
    time_end      (ID) datetime64[ns] 35kB dask.array<chunksize=(4359,), meta=np.ndarray>
    merge_ledger  (time, ID, sibling_ID) int32 229MB dask.array<chunksize=(1, 4359, 12), meta=np.ndarray>
Attributes: (12/15)
    allow_merging:               1
    N_objects_prefiltered:       138734
    N_objects_filtered:          34034
    N_events_final:              4359
    R_fill:                      32
    T_fill:                      2
    ...                          ...
    preprocessed_area_fraction:  0.22705580383575777
    overlap_threshold:           0.5
    nn_partitioning:             1
    total_merges:                9404
    multi_parent_merges:         2634
    marex_staging_dir:           /scratch/b/b382615/mhws/TEMP/marex_stage_233...

In [8]:
merges_ds

<xarray.Dataset> Size: 1MB
Dimensions:        (merge_ID: 9404, parent_idx: 12, child_idx: 12)
Dimensions without coordinates: merge_ID, parent_idx, child_idx
Data variables:
    parent_IDs     (merge_ID, parent_idx) int32 451kB 49 50 57 -1 ... -1 -1 -1
    child_IDs      (merge_ID, child_idx) int32 451kB 82 34035 34036 ... -1 -1 -1
    overlap_areas  (merge_ID, parent_idx) float32 451kB 2.833e+12 ... -1.0
    merge_time     (merge_ID) datetime64[ns] 75kB 2012-01-03T23:59:59 ... 201...
    n_parents      (merge_ID) int8 9kB 3 2 2 2 2 4 2 2 2 2 ... 2 2 2 2 2 2 2 2 2
    n_children     (merge_ID) int8 9kB 3 2 2 2 2 4 2 2 2 2 ... 2 2 2 2 2 2 2 2 2
Attributes:
    fill_value:  -1

In [9]:
# Save IDed/Tracked/Merged Events to `zarr` for more efficient parallel I/O

file_name = scratch_dir / "mhws" / "extreme_events_merged_unstruct.zarr"
extreme_events_ds.to_zarr(file_name, mode="w")

In [10]:
# Save Merges Dataset to netcdf (as in the gridded example -- the merge ledger is
# the record of which events split/merged, and is not recoverable from the ID field)

file_name = scratch_dir / "mhws" / "extreme_events_merged_unstruct_merges.nc"
merges_ds.to_netcdf(file_name, mode="w")

# Under `compute_mode="streaming"` the returned dataset reads LAZILY from `temp_dir`, so the
# staging directory deliberately outlives `run()`. Now that the results are on disk, release
# it -- the second tracker below stages its own copy, and each is ~230 GB at this length.
# (The `atexit` backstop does not survive a wall-clock kill, so do not rely on it.)
marEx.clear_staging(extreme_events_ds)

2026-08-22 13:02:29 - marEx.detect.compute_mode - DEBUG - [PID:2330325] - clear_staging:115 - Cleared staging directory: /scratch/b/b382615/mhws/TEMP/marex_stage_2330325_b767d5c3


### Use Centroid-based Partitioning Method for Comparison

In [11]:
# Run ID, Tracking, & Merging

tracker = marEx.tracker(
    ds.extreme_events,
    ds.mask,
    area_filter_absolute=13500,  # Keep only objects of at least 13500 cells (~328,000 km2 at this mesh's 24.34 km2 mean cell area). An absolute floor rather than a quartile: at 14.9M cells the smallest-80% cut still leaves tens of thousands of fragments, which fragment merge events rather than resolving them.
    R_fill=32,  # Fill small holes with radius < 32 elements, i.e. ~158 km (32 x the 4.93 km mean cell spacing),
    T_fill=2,  # Allow gaps of 2 days and still continue the event tracking with the same ID
    allow_merging=True,  # Allow extreme events to split/merge. Keeps track of merge events & unique IDs.
    overlap_threshold=0.5,  # Overlap threshold for merging events. If overlap < threshold, events keep independent IDs.
    nn_partitioning=False,  # Use old Centroid-based partitioning method (Di Sun et al. 2023).
    temp_dir=str(scratch_dir / "mhws" / "TEMP/"),  # Temporary Scratch Directory for Dask
    verbose=True,  # Enable detailed logging
    # -- Unstructured Grid Options --
    unstructured_grid=True,  # Use Unstructured Grid
    dimensions={"x": "ncells"},  # Need to tell MarEx the new Unstructured dimension
    coordinates={"x": "lon", "y": "lat"},  # Coordinates for Unstructured Grid
    neighbours=ds.neighbours,  # Connectivity array for the Unstructured Grid Cells
    cell_areas=ds.cell_areas,  # Cell areas for each Unstructured Grid Cell
    compute_mode="streaming",  # Stage every whole-field intermediate to zarr under `temp_dir` rather than pinning it in cluster RAM. Required at this length: one int32 whole field is 1096 x 14.9M x 4 B = 65.3 GB, and `persist` holds several at once.
)

extreme_events_ds, merges_ds = tracker.run(return_merges=True)
extreme_events_ds

2026-08-22 13:02:29 - marEx - INFO - [PID:2330325] - configure_logging:177 - MarEx logging configured - Level: DEBUG, Mode: verbose


2026-08-22 13:02:29 - marEx.track.tracker - INFO - [PID:2330325] - __init__:481 - Initialising MarEx tracker


2026-08-22 13:02:29 - marEx.track.tracker - INFO - [PID:2330325] - __init__:482 - Grid type: unstructured


2026-08-22 13:02:29 - marEx.track.tracker - INFO - [PID:2330325] - __init__:483 - Parameters: R_fill=32, T_fill=2, area_filter_quartile=None, area_filter_absolute=13500


2026-08-22 13:02:29 - marEx.track.tracker - DEBUG - [PID:2330325] - __init__:487 - Tracking options: allow_merging=True, nn_partitioning=False, overlap_threshold=0.5


...

2026-08-22 14:34:20 - marEx.track.tracker - INFO - [PID:2330325] - run:928 - Tracking pipeline completed successfully - 4305 events identified


2026-08-22 14:34:20 - marEx.track.tracker - DEBUG - [PID:2330325] - run:929 - Final dataset dimensions: FrozenMappingWarningOnValuesAccess({'time': 1096, 'ncells': 14886338, 'ID': 4305, 'component': 2, 'sibling_ID': 12})


2026-08-22 14:34:21 - marEx.track.tracker - DEBUG - [PID:2330325] - log_memory_usage:287 - Pipeline completion - Memory Usage - RSS: 3257.9MB, Virtual: 67443.5MB, Percent: 0.3%, Available: 984249.3MB


2026-08-22 14:34:21 - marEx.track.tracker - DEBUG - [PID:2330325] - run:941 - Returning both events and merge datasets


<xarray.Dataset> Size: 66GB
Dimensions:       (time: 1096, ncells: 14886338, ID: 4305, component: 2,
                   sibling_ID: 12)
Coordinates:
  * time          (time) datetime64[ns] 9kB 2012-01-01T23:59:59 ... 2014-12-3...
  * ID            (ID) int32 17kB 1 2 3 4 5 6 ... 4300 4301 4302 4303 4304 4305
  * component     (component) int64 16B 0 1
    lat           (ncells) float64 119MB dask.array<chunksize=(14886338,), meta=np.ndarray>
    lon           (ncells) float64 119MB dask.array<chunksize=(14886338,), meta=np.ndarray>
Dimensions without coordinates: ncells, sibling_ID
Data variables:
    ID_field      (time, ncells) int32 65GB dask.array<chunksize=(1, 14886338), meta=np.ndarray>
    global_ID     (time, ID) int32 19MB dask.array<chunksize=(1, 4305), meta=np.ndarray>
    area          (time, ID) float32 19MB dask.array<chunksize=(1, 4305), meta=np.ndarray>
    centroid      (component, time, ID) float32 38MB dask.array<chunksize=(1, 1, 4305), meta=np.ndarray>
    presence      (time, ID) bool 5MB dask.array<chunksize=(1, 4305), meta=np.ndarray>
    time_start    (ID) datetime64[ns] 34kB dask.array<chunksize=(4305,), meta=np.ndarray>
    time_end      (ID) datetime64[ns] 34kB dask.array<chunksize=(4305,), meta=np.ndarray>
    merge_ledger  (time, ID, sibling_ID) int32 226MB dask.array<chunksize=(1, 4305, 12), meta=np.ndarray>
Attributes: (12/15)
    allow_merging:               1
    N_objects_prefiltered:       138734
    N_objects_filtered:          34034
    N_events_final:              4305
    R_fill:                      32
    T_fill:                      2
    ...                          ...
    preprocessed_area_fraction:  0.22705580383575777
    overlap_threshold:           0.5
    nn_partitioning:             0
    total_merges:                9872
    multi_parent_merges:         2515
    marex_staging_dir:           /scratch/b/b382615/mhws/TEMP/marex_stage_233...

In [12]:
# Save IDed/Tracked/Merged Events to `zarr` for more efficient parallel I/O

file_name = scratch_dir / "mhws" / "extreme_events_merged_centroid_unstruct.zarr"
extreme_events_ds.to_zarr(file_name, mode="w")

# Under `compute_mode="streaming"` the returned dataset reads LAZILY from `temp_dir`, so the
# staging directory deliberately outlives `run()`. Now that the results are on disk, release
# it -- the second tracker below stages its own copy, and each is ~230 GB at this length.
# (The `atexit` backstop does not survive a wall-clock kill, so do not rely on it.)
marEx.clear_staging(extreme_events_ds)

2026-08-22 14:35:38 - marEx.detect.compute_mode - DEBUG - [PID:2330325] - clear_staging:115 - Cleared staging directory: /scratch/b/b382615/mhws/TEMP/marex_stage_2330325_e2c7008a
